# Chapter 7 — Learning Systems for Neural Data

**Book:** *Brain–Computer Interfaces and Cognitive Computing: An Introduction*  
**Prerequisites:** Chapters 0–6.  
**Author:** Er. Rishabh Aryan

Stage S5 is a function $f_\theta$ from features $z$ (and optional context $h$) to a decision. This chapter connects the machine-learning course you already took to the constraints Chapters 4–6 imposed: few trials, drifting sources, overlapping windows, and a user who will notice a late error more than a pretty ROC curve.

Deep models are admitted. They are not a solvent. A network trained on a leaked split is still a leaked split.

**Learning outcomes**

1. Fit and evaluate a linear decoder with regularisation and a causal split.
2. Explain why neural trials are not i.i.d. images, and what that does to cross-validation.
3. Locate convolutional, recurrent, and sequence models in the loop without treating them as a different science.
4. Separate calibration, domain shift, and foundation-style pooling across subjects.
5. Report a result with chance level, a baseline, and an artefact audit, or refuse to report it.


---

## 7.1 The learning problem, stated honestly

Each trial or window yields a pair $(z, y)$. $z \in \mathbb{R}^d$ comes from $\Phi$ of Chapter 6. $y$ is a class, a character, a kinematic vector, a token, or a scalar state.

We choose $\theta$ to make $f_\theta(z)$ close to $y$ on data the loop has not yet seen. “Close” is not unique:

- classification error or balanced accuracy, when $y$ is a class;
- information transfer, when the user faces an alphabet (Chapter 9);
- word error or listener comprehension, when $y$ is speech;
- calibration of a posterior, when Stage S6 will multiply by a prior.

> **Definition 7.1 (Decoder).**  
> A parametric map $f_\theta$ from representation and optional context to a decision variable used at Stage S5.

> **Definition 7.2 (Baseline).**  
> A simpler $f$ evaluated on the *same* split. A linear decoder on the same $z$ is the default baseline for any network. Chance level is not a baseline; it is a floor.


---

## 7.2 Linear methods still carry the field

For many scalp paradigms, after a good $\Phi$, a linear map is enough:

$$
\hat{y} = \mathrm{sign}(w^\top z + b)
$$

or, for scores used by S6,

$$
s = w^\top z + b.
$$

**Linear discriminant analysis (LDA)** uses class means and a pooled covariance. When $d$ approaches the number of trials, the covariance is singular. **Regularised LDA** shrinks that covariance toward a diagonal or toward a scaled identity. That shrinkage *is* the method, not a footnote.

**Logistic regression** and a **linear SVM** differ in loss. They agree that the decision is a hyperplane in feature space (Chapter 0). Start here. If a network cannot beat regularised LDA on a leakage-safe split, the network has not earned the paragraph.

**Ridge / linear regression** remains the workhorse for continuous kinematics.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})


def make_blocks(n_blocks=8, n_per=40, d=12, drift=0.15):
    '''Two classes; mean slowly drifts across blocks (session non-stationarity).'''
    Z, y, block = [], [], []
    mu0 = np.zeros(d)
    mu1 = np.zeros(d)
    mu1[:3] = 1.1
    for b in range(n_blocks):
        shift = drift * b * np.linspace(0, 1, d)
        for _ in range(n_per):
            lab = np.random.randint(0, 2)
            mu = (mu0 if lab == 0 else mu1) + shift
            Z.append(mu + np.random.randn(d))
            y.append(lab)
            block.append(b)
    return np.array(Z), np.array(y), np.array(block)


def rlda_fit(Z, y, shrink=0.3):
    Z0, Z1 = Z[y == 0], Z[y == 1]
    m0, m1 = Z0.mean(0), Z1.mean(0)
    S = np.cov(np.vstack([Z0, Z1]), rowvar=False)
    S = (1 - shrink) * S + shrink * np.eye(S.shape[0]) * np.trace(S) / S.shape[0]
    w = np.linalg.solve(S, m1 - m0)
    b = -0.5 * w @ (m0 + m1)
    return w, b


def predict(Z, w, b):
    return (Z @ w + b > 0).astype(int)


Z, y, block = make_blocks()
# Honest: train early blocks, test later blocks
train, test = block <= 4, block >= 5
w, b = rlda_fit(Z[train], y[train])
acc_time = np.mean(predict(Z[test], w, b) == y[test])

# Leaky: shuffle all trials
idx = np.random.permutation(len(y))
cut = int(0.6 * len(y))
w_l, b_l = rlda_fit(Z[idx[:cut]], y[idx[:cut]])
acc_leak = np.mean(predict(Z[idx[cut:]], w_l, b_l) == y[idx[cut:]])

print(f"causal block split accuracy  = {acc_time:.3f}")
print(f"shuffled trial split accuracy= {acc_leak:.3f}")
print("chance = 0.500")
print("The shuffled number is the one posters prefer. It is not the interface.")


Re-run with `drift=0` and watch the two accuracies meet. The gap *is* non-stationarity. No architecture removes the need to measure that gap.


---

## 7.3 Why neural examples are not ImageNet

Standard ML intuition assumes i.i.d. draws. Neural windows violate that in several simultaneous ways.

- **Temporal correlation.** Adjacent windows share samples and share state.  
- **Block structure.** Fatigue, gel drying, instruction changes.  
- **Label noise.** “Imagined left hand” is an instruction, not a measured generator. Some trials contain no ERD.  
- **Few shots.** Tens to hundreds of trials, not millions of images.  
- **User identity.** $P(z \mid y, \text{user})$ is a different distribution for each skull.  
- **Feedback.** In a closed loop the user’s $z$ depends on yesterday’s $f_\theta$.

Cross-validation that ignores these produces a number that predicts a poster, not Tuesday.

**Permitted splits, in order of severity.**

1. Later blocks of the same session (minimum).  
2. Later sessions of the same user.  
3. Held-out users (subject-independent claim).  

State which claim you are making. A model that needs Amina’s morning data is not “zero-cal.”


---

## 7.4 Beyond linear: when and why

**Riemannian / covariance methods (outline).**  
For imagery, a trial can be a covariance matrix. Distances on the manifold of symmetric positive-definite matrices, followed by a linear classifier in a tangent space, are strong baselines on many EEG benchmarks. Learn the idea: the object is $\Sigma$, not a long vector of raw samples. Implementation can wait for a methods elective.

**Kernel methods.** Historically important. Usually replaced either by better $\Phi$ plus linear $f$, or by networks.

**Generative models of the ERP or of rhythms.** Useful when data are scarce and the paradigm is simple. They are priors on $z$, not on Amina’s mind.

Move to non-linear $f$ when you can name the structure linear $f$ cannot see: long phonetic sequences, mixed selectivity across a large array, cross-subject invariances you can *test* on held-out users.


---

## 7.5 Networks as $\Phi_{\text{feat}}$ and as $f_\theta$

A convolutional stack on multi-channel time is a learned spatial-temporal filter bank. A recurrent or temporal-convolution stack is a learned $h_t$. A transformer over windows is a learned context. None of these changes Definition 7.1.

**Discipline.**

- The network is trained on the training split only. Early stopping uses a *later* validation block, not a shuffle.  
- Input windows obey Chapter 6 causality.  
- Parameter count must face trial count. A million-parameter net on eighty trials will memorise gel patterns.  
- Report the linear baseline on the same $\Phi$ *and* a linear baseline on the network’s input.  

**Sequence models for speech.**  
Attempted-speech systems output tokens. Connectionist temporal classification, encoder–decoder models, and language-model fusion live at S5–S6. Ablate the language model (Exercise 5.5) or you are reporting English, not cortex.

**Calibration after a network.**  
A large model trained on many subjects still meets Amina as a new domain. A small adapter, a new last layer, or a few minutes of labelled attempt may remain. “Foundation” does not mean “no Tuesday.”


---

## 7.6 Foundation-style models, without the press release

Pooling recordings across people and days to pretrain a representation is a legitimate research programme. It is the analogue of pretraining elsewhere in machine learning.

What it does **not** currently justify in an introductory textbook:

- that mental privacy is solved because the model is generic;  
- that calibration burden is zero;  
- that a held-out *task* will work because held-out *subjects* worked on another task;  
- that the representation is a theory of cognition.

> **Definition 7.3 (Subject-independent claim).**  
> A performance number obtained with entire users held out of every stage of $\Phi$ and $f$, including normalisation statistics.

If channel picks, component rejection, or early stopping used those users, the claim is false.


---

## 7.7 Adaptation during the life of the loop

Chapter 4 defined calibration burden. Learning methods implement it.

**Static $\theta$.** Fit once. Honest only if you measure later-block decay.

**Scheduled recalibration.** New labelled trials. Counts against Amina’s morning.

**Supervised incremental updates.** Use explicit confirms and undos as labels. The undo key is part of the system.

**Unsupervised or semi-supervised updates.** Tempting; error reinforcement is the failure mode. If you use decoder outputs as labels, report the collapse when the first bias appears.

Cognitive computing contributes a *policy*: when to request labels, when to freeze $\theta$, when to slow the alphabet. It does not exempt you from Definition 7.3 if you claimed subject independence.


---

## 7.8 Metrics that survive Chapter 4

For each result paragraph, include:

- the split (Definition 7.3 or a weaker named split);  
- chance level and a linear baseline;  
- sample size in *independent* blocks, not overlapping windows;  
- closed-loop metric if you claim an interface (corrected letters per minute, understood words);  
- artefact-audit outcome (Chapter 6);  
- calibration minutes.

Receiver-operating curves on leaked windows are decorative.

Class imbalance: a P300 rare class is rare. Accuracy can be 0.94 while the decoder never finds the target. Report balanced error or a proper scoring rule.


In [ ]:
def balanced_accuracy(y_true, y_hat):
    ba = []
    for c in np.unique(y_true):
        mask = y_true == c
        ba.append(np.mean(y_hat[mask] == c))
    return float(np.mean(ba))


# Imbalanced toy: 90 percent class 0
y_true = np.array([0] * 90 + [1] * 10)
y_hat = np.zeros_like(y_true)  # majority classifier
print("accuracy           ", np.mean(y_true == y_hat))
print("balanced accuracy  ", balanced_accuracy(y_true, y_hat))
print("A P300 decoder that never detects a rare flash looks like the first number.")


---

## 7.9 A worked miniature

Suppose Amina’s imagery study yields eight blocks. You will:

1. freeze $\Phi$ structure (bands, window, montage) from literature, not from her test blocks;  
2. estimate CSP and regularised LDA on blocks 1–5;  
3. optionally peek at block 6 once to choose shrinkage, then lock $\theta$;  
4. report blocks 7–8 once;  
5. train a nuisance EMG decoder on the same split;  
6. if you add a small network, it must beat step 4 on the same locked test blocks.

Anything more baroque needs more data, not more adjectives.


---

## 7.10 What to carry into Chapter 8

Chapter 8 returns to cognitive computing with S5 now specified.

1. $f_\theta$ is ordinary statistical learning under hostile dependence.  
2. Linear regularised decoders remain the baseline of record.  
3. Networks are permitted when they beat that baseline on a named split.  
4. Subject-independent and foundation claims have a definition; most posters do not meet it.  
5. Adaptation is part of $f$, and part of Amina’s day.


---

## 7.11 Chapter summary

Learning a BCI decoder is supervised prediction with temporal dependence, label noise, few trials, and a drifting generator. Regularised linear methods on designed features still carry much of the field. Covariance-based and sequence models earn their place by beating those methods on splits that do not shuffle overlapping windows. Cross-user pretraining is a programme with a precise claim, not a synonym for intelligence. Report chance, baseline, split, calibration time, and an artefact audit. The rest is demonstration.


---

## Exercises

**7.1** Using the first cell, vary `drift` in $\{0, 0.05, 0.15, 0.3\}$ and tabulate causal versus shuffled accuracy. Write four sentences on what posters hide.

**7.2** Why is chance level not a baseline (Definition 7.2)? Give a sentence a thesis examiner would accept.

**7.3** You have 60 imagery trials and a network with $8 \times 10^5$ parameters. Argue for or against training it. If against, name the model you will use instead.

**7.4** Write a subject-independent protocol that satisfies Definition 7.3 for 12 users. List every statistic that must be refit inside the training users only.

**7.5** A speech decoder plus a language model reaches 2 percent word error. Describe two ablations that isolate S5 from S6.

**7.6** Propose an incremental-update rule that uses Amina’s undo key. State the failure mode if she stops undoing when tired.

**7.7** Compute balanced accuracy versus accuracy for a P300 flash decoder that flags 5 percent of flashes as targets when 8 of 36 flashes are true targets and the decoder never hits them. (Construct a small table.)

**7.8** Take any recent “deep learning BCI” abstract. Rewrite its performance sentence so that it could appear in this chapter. If you cannot, the abstract did not contain a result.


---

## Annotated reading

- Blankertz and colleagues on regularised LDA for EEG — the baseline of record.  
- Lotte et al., reviews of ML for BCI — use to see that networks are late arrivals, not replacements.  
- A Riemannian BCI tutorial — after you can explain why $\Sigma$ is a feature.  
- One contemporary speech-BCI paper and one sceptical evaluation checklist; read them on the same afternoon.

---

*End of Chapter 7.*  
*Next: Chapter 8 — Cognitive computing in this setting.*
